In [1]:
import re
import requests
import numpy as np
import pandas as pd

In [45]:
REGISTRY = pd.read_csv("datasets/ClearedRegistry.csv")
BASE_URL = "https://bo.nalog.gov.ru/search?query=" # 5038038838

In [18]:
registry = REGISTRY
registry.sample(2)

,Наименование / ФИО,Тип субъекта,Категория,ИНН,Основной вид деятельности,Регион,Город,Населенный пункт,Дата включения в реестр,Наличие лицензий,Среднесписочная численность работников за предшествующий календарный год
4619,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""РУСС...",Юридическое лицо,Малое предприятие,5029208473,41.20 Строительство жилых и нежилых зданий,77 - г.Москва,NaN,NaN,10.11.2022,Нет,2.0
2055,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""ЖИЛС...",Юридическое лицо,Малое предприятие,5403173051,41.20 Строительство жилых и нежилых зданий,54 - Новосибирская область,г Новосибирск,NaN,01.08.2016,Нет,2.0


In [22]:
search_queries: dict[str, str] = {}
def get_company_search_queries(row: pd.Series) -> None:
    search_queries[int(row["ИНН"])] = BASE_URL + str(row["ИНН"])

In [23]:
registry.apply(get_company_search_queries, axis=1)
len(search_queries)

8927

In [36]:
from time import time, sleep
from selenium import webdriver
from selenium.webdriver.common.by import By

In [ ]:
driver = webdriver.Chrome()

In [35]:
def get_profile_links(search_queries: dict[str, str]):
    enns = search_queries.keys()
    profile_links: dict[int, str]
    for enn in enns:
        driver.get(search_queries[enn])
        root = driver.find_element(By.CSS_SELECTOR, "#root")
        while True:
            try:
                profile_links[enn] = root.find_element(By.CSS_SELECTOR,
                                                       "main > div > div > div.results-search-table\
                                                        > div.results-search-tbody > a"
                                                      ).get_attribute("href")
                break
            except Exception:
                if "не найдена" in root.text.lower():
                    break
    return profile_links

In [ ]:
profile_links = get_profile_links(search_queries)

In [41]:
def calc_of_ebit(report: dict[int, str]) -> dict[str, int | None]:
    ebit = 0
    new_report = {"ebit": np.nan, "receivable": np.nan, "payable": np.nan}

    if report.get(2300) and report[2300] != "-": # income before taxes
        ebit += int(report[2300]) if "(" not in report[2300] else -int(report[2300][1:-1])
    elif report.get(2400) and report.get(2410): # net income and taxes
        if "-" in report.get(2400)+report.get(2410): return new_report
        n_i = int(report[2400]) if "(" not in report[2400] else -int(report[2400][1:-1])
        i_t = int(report[2410]) if "(" not in report[2410] else int(report[2410][1:-1])
        ebit += n_i + i_t
    else: return new_report

    if report.get(2320) and report[2320] != "-": # interest receivable
        new_report["receivable"] = int(report[2320]) if "(" not in report[2320] else 0
        ebit -= new_report["receivable"]
    elif not report.get(2320): new_report["receivable"] = 0

    if report.get(2330) and report[2330] != "-": # interest payable
        new_report["payable"] = int(report[2330][1:-1])
        ebit += int(report[2330][1:-1])
    elif not report.get(2330): new_report["payable"] = 0

    new_report["ebit"] = ebit
    return new_report

In [42]:
def extracting_ebit(report_table) -> int:
    report = {}
    new_report = {}
    rows = report_table.find_elements(By.CLASS_NAME, 'tabulator-tree-level-0')
    for row in rows:
        code  = row.find_elements(By.CSS_SELECTOR, '[tabulator-field="code"]')
        value = row.find_elements(By.CSS_SELECTOR, '[tabulator-field="current"]')
        if len(code) + len(value) < 2: continue
        for i in range(len(code)):
            try:
                code  = int(code[i].text)
                value =    value[i].text
                if "\n" in value: value = value[:value.find("\n")]
                report[int(code)] = value.replace(" ", "")
            except Exception: pass
    return calc_of_ebit(report)

In [43]:
def get_report():
    while True:
        try:
            driver.find_element(By.CSS_SELECTOR, "#financialResult > button").click()
            break
        except Exception: pass

    # financial results section
    report_selector = "#root > main > div.grid-reports > div.grid-reports-body > div > \
                    div.grid-reports-items > div.grid-reports-item.is-open > \
                    div.ReactCollapse--collapse > div > div > div.tabulator > \
                    div.tabulator-tableHolder > div"

    begin, timer = time(), 0
    report = driver.find_element(By.CSS_SELECTOR, report_selector)
    while "2400" not in report.text and timer < 3.5:
        report = driver.find_element(By.CSS_SELECTOR, report_selector)
        timer = time() - begin
    return extracting_ebit(report)

In [44]:
def get_EBITs(profile_links: dict[str, str]):
    cnt = 0
    enns = profile_links.keys()^(profile_links.keys()&ebits.index.tolist())
    
    for enn in enns:
        try: driver.get(profile_links[enn])
        except Exception: continue
        if "не найдена" in driver.page_source: continue
            
        ebit: dict[str, int] = dict()
        nyears = 0 # количество годовых отчётностей
        while not nyears:
            nyears = len(driver.find_elements(By.CSS_SELECTOR,
                    f"#root > main > div.grid-reports > div.grid-reports-header-fixed > div > div > \
                      div.grid-reports-header > div > div.grid-reports-header-row.grid-reports-header-top > \
                      div.grid-reports-header-top__period > button"))

        # обход каждого года
        for i in range(2, 2+nyears):

            year, report_exists = "", False
            begin, timer = time(), 0
            while True:
                try:
                    year_button = driver.find_element(By.CSS_SELECTOR,
                                     f"#root > main > div.grid-reports > div.grid-reports-header-fixed > div > div > \
                                     div.grid-reports-header > div > div.grid-reports-header-row.grid-reports-header-top > \
                                     div.grid-reports-header-top__period > button:nth-child({i})")
                    year = year_button.text
                    year_button.click()
                    break
                except Exception: pass

            while not report_exists and timer < 2.5:
                try:
                    report_exists = "ФИНАНСОВЫЕ РЕЗУЛЬТАТЫ" in driver.find_element(By.CSS_SELECTOR,
                            "#root > main > div.grid-reports > div.grid-reports-header-fixed >\
                            div > div > div.grid-reports-header > div > \
                            div.grid-reports-header-row.grid-reports-header-buttons").text
                except Exception:
                    timer = time() - begin
            if not report_exists: continue

            report = get_report()
            if year and report: ebit[year] = report 
            driver.execute_script("window.scrollTo(0, 0)")

        cnt += 1
        # добавление ebit в таблицу ebits
        for year in ebit:
            for metric in ebit[year]:
                ebits.loc[int(enn), year+metric] = ebit[year][metric]

        if not cnt%4: ebits.to_csv("datasets/EBITs.csv")
        try:
            print(f"---{cnt}\t{enn}: {profile_links[enn]}")
            print(ebits.loc[int(enn)])
        except Exception: pass

In [ ]:
get_EBITs(profile_links)

In [46]:
ebits.sample(10)

,2021ebit,2021payable,2021receivable,2022ebit,2022payable,2022receivable,2023ebit,2023payable,2023receivable,2024ebit,2024payable,2024receivable,2025ebit,2025payable,2025receivable,2026ebit,2026receivable,2026payable
ENN,,,,,,,,,,,,,,,,,,
6452117855,8060.0,NaN,18.0,2137.0,NaN,16.0,9177.0,NaN,2069.0,32746.0,NaN,3128.0,8436.0,0.0,11242.0,NaN,NaN,NaN
5904062514,6954.0,1614.0,NaN,15609.0,874.0,NaN,12888.0,1429.0,555.0,19563.0,792.0,4419.0,30574.0,0.0,0.0,NaN,NaN,NaN
5835042056,-24204.0,2.0,268.0,-434.0,0.0,0.0,9333.0,NaN,23.0,20187.0,NaN,0.0,-74937.0,828.0,0.0,NaN,NaN,NaN
2311245889,117897.0,4351.0,2794.0,243551.0,2203.0,294.0,90858.0,3541.0,2686.0,-27606.0,19349.0,8151.0,-844.0,19200.0,0.0,NaN,NaN,NaN
7718169528,-1789.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278153546,2493.0,NaN,NaN,19895.0,NaN,NaN,17002.0,NaN,632.0,95011.0,NaN,13529.0,23249.0,0.0,0.0,NaN,NaN,NaN
5503239034,4478.0,NaN,NaN,2897.0,NaN,NaN,3687.0,NaN,NaN,1351.0,NaN,NaN,3651.0,3088.0,0.0,NaN,NaN,NaN
5190063492,968.0,NaN,0.0,-3761.0,0.0,0.0,508.0,NaN,0.0,4555.0,NaN,0.0,-3348.0,0.0,0.0,NaN,NaN,NaN
2315119910,-16685.0,NaN,NaN,3500.0,NaN,NaN,-3392.0,NaN,0.0,-3934.0,NaN,0.0,-10047.0,0.0,0.0,NaN,NaN,NaN
